In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [5]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.2 MB/s eta 0:00:00


In [6]:
import nltk
from nltk.tokenize import word_tokenize
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import gensim.downloader as api
import re

In [7]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [11]:
import pandas as pd
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [12]:
X = df['review']
Y = df['sentiment'].map({"negative": 0, "positive": 1})

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove special characters
    tokens = word_tokenize(text)
    return tokens

In [14]:
X_train = X.apply(clean_text)
X_test = X.apply(clean_text)

In [15]:
word2vec = api.load("word2vec-google-news-300")
embedding_dim = 300

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [16]:
def get_embedding(tokens, embedding_dim=300):
    vectors = [word2vec[word] for word in tokens if word in word2vec]
    if len(vectors) == 0:
        return np.zeros(embedding_dim)
    return np.mean(vectors, axis=0)

X_train = X_train.apply(lambda x: get_embedding(x))
X_test = X_test.apply(lambda x: get_embedding(x))

In [17]:
class ReviewData(Dataset):
    def __init__(self, X_data, Y_data):
        self.reviews = torch.tensor(np.array(X_data.tolist()), dtype=torch.float32)
        self.labels = torch.tensor(Y_data.tolist(), dtype=torch.float32)

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, item):
        return self.reviews[item], self.labels[item]

train_dataset = ReviewData(X_train, Y)
test_dataset = ReviewData(X_test, Y)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [18]:
class Sentiment_model(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 150, batch_first=True)
        self.fc = nn.Linear(150, 1)  # output for binary classification

    def forward(self, x):
        embedded = self.embedding(x)  # shape: (batch, seq_len, 100)
        _, (final_hidden_state, _) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))
        output = torch.sigmoid(output)  # output between 0 and 1
        return output


In [21]:
model = Sentiment_model(embedding_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    total_loss = 0
    correct_train = 0
    total_train = 0

    model.train()
    for reviews, labels in train_loader:
        reviews = reviews.long().to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(reviews)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    train_accuracy = (correct_train / total_train) * 100

    # Evaluation
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for reviews, labels in test_loader:
            reviews = reviews.long().to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(reviews)
            predicted = (outputs > 0.5).float()
            correct_test += (predicted == labels).sum().item()
            total_test += labels.size(0)

    test_accuracy = (correct_test / total_test) * 100

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader):.4f}, "
          f"Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")


Epoch 1, Loss: 0.6936, Train Acc: 49.79%, Test Acc: 50.00%
Epoch 2, Loss: 0.6932, Train Acc: 50.32%, Test Acc: 50.00%
Epoch 3, Loss: 0.6932, Train Acc: 49.99%, Test Acc: 50.00%
Epoch 4, Loss: 0.6932, Train Acc: 49.64%, Test Acc: 50.00%
Epoch 5, Loss: 0.6932, Train Acc: 49.83%, Test Acc: 50.00%
Epoch 6, Loss: 0.6932, Train Acc: 50.22%, Test Acc: 50.00%
Epoch 7, Loss: 0.6932, Train Acc: 49.95%, Test Acc: 50.00%
Epoch 8, Loss: 0.6932, Train Acc: 50.26%, Test Acc: 50.00%
Epoch 9, Loss: 0.6932, Train Acc: 50.06%, Test Acc: 50.00%
Epoch 10, Loss: 0.6932, Train Acc: 49.98%, Test Acc: 50.00%


In [31]:
def predict_sentiment(model, review_text):
    tokens = clean_text(review_text)
    review_vector = get_embedding(tokens)  # make sure this returns a list of integers (token indices)

    # Convert to LongTensor for embedding layer
    review_tensor = torch.tensor(review_vector, dtype=torch.long).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(review_tensor)

    sentiment = "Positive" if output.item() > 0.5 else "Negative"
    return sentiment, output.item()

sample_review = "An emotional rollercoaster that left me in tears!"
sentiment, confidence = predict_sentiment(model, sample_review)
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")


Predicted Sentiment: Positive (Confidence: 0.5032)
